# SurahChain Pre-training على Google Colab

هذا الدفتر:
1. يثبّت المتطلبات
2. يستنسخ المستودع
3. يسحب بيانات Pre-training (بدون CKG)
4. يدرّب SurahChain مع دعم **resume**
5. يرفع الـcheckpoints تلقائياً إلى GitHub بعد الانتهاء

**قبل التشغيل:** Runtime → Change runtime type → **GPU (T4)**

## 0) الإعدادات — عدّل هنا فقط

In [ ]:
# ===== إعدادات المستخدم =====
GITHUB_TOKEN = ""  # الصق Personal Access Token هنا (repo scope) — لا تشاركه علناً
REPO = "aliahmed369000000-ai/Neural-Service-Mesh"
BRANCH = "main"

# بيانات وتدريب
SCN_N = 30000          # عدد المقاطع
SCN_EPOCHS = 10        # حقب هذه الجولة (عند resume = إضافية)
SCN_D_MODEL = 128      # 128 آمن | 256 إن سمحت الذاكرة
SCN_BATCH = 32         # على GPU جرّب 32 أو 64
SCN_LR = 1e-3
SCN_FRESH = False      # True = ابدأ من الصفر وتجاهل checkpoint

# رفع النتائج
AUTO_PUSH = True
COMMIT_MESSAGE = "Colab: SurahChain pretrain checkpoints + state"

print("الإعدادات جاهزة")
print(f"N={SCN_N} EPOCHS={SCN_EPOCHS} D_MODEL={SCN_D_MODEL} BATCH={SCN_BATCH} FRESH={SCN_FRESH}")

## 1) فحص GPU وتثبيت الحزم

In [ ]:
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ لا يوجد GPU — التدريب سيكون أبطأ. فعّل GPU من Runtime.")

!pip -q install datasets huggingface_hub

## 2) استنساخ المستودع

In [ ]:
import os
from pathlib import Path

WORK = Path("/content/Neural-Service-Mesh")
if not GITHUB_TOKEN:
    raise SystemExit("ضع GITHUB_TOKEN في خلية الإعدادات (مطلوب للاستنساخ والرفع)")

clone_url = f"https://{GITHUB_TOKEN}@github.com/{REPO}.git"
if WORK.exists():
    print("المستودع موجود — git pull")
    %cd /content/Neural-Service-Mesh
    !git remote set-url origin {clone_url}
    !git pull origin {BRANCH}
else:
    %cd /content
    !git clone --depth 1 -b {BRANCH} {clone_url} Neural-Service-Mesh
    %cd /content/Neural-Service-Mesh

!git config user.email "nsm-bot@users.noreply.github.com"
!git config user.name "NSM Bot"
print("cwd:", os.getcwd())

## 3) تحضير بيانات Pre-training (حتى SCN_N)

In [ ]:
import os
os.environ["SCN_N"] = str(SCN_N)
os.environ["SCN_FORCE_REBUILD"] = "0"

!python experiments/surah_chain_network/prepare_pretrain_data.py

## 4) التدريب (مع resume تلقائي إن وُجد checkpoint)

In [ ]:
import os
os.environ["SCN_N"] = str(SCN_N)
os.environ["SCN_EPOCHS"] = str(SCN_EPOCHS)
os.environ["SCN_D_MODEL"] = str(SCN_D_MODEL)
os.environ["SCN_BATCH"] = str(SCN_BATCH)
os.environ["SCN_LR"] = str(SCN_LR)
os.environ["SCN_FRESH"] = "1" if SCN_FRESH else "0"

!python experiments/surah_chain_network/train_pretrain_torch.py

## 5) رفع النتائج إلى GitHub

In [ ]:
import os
from pathlib import Path

if not AUTO_PUSH:
    print("AUTO_PUSH=False — تخطّي الرفع")
else:
    exp = Path("experiments/surah_chain_network")
    files = [
        exp / "checkpoints" / "best_pretrain_torch.pt",
        exp / "checkpoints" / "latest_pretrain_torch.pt",
        exp / "checkpoints" / "pretrain_torch_state.json",
        exp / "tokenizer_vocab_pretrain.json",
    ]
    for f in files:
        if f.exists():
            print("+", f, f.stat().st_size)
            # checkpoints مستثناة بـ gitignore → add -f
            !git add -f {f}
        else:
            print("missing:", f)

    !git status --short
    !git commit -m "{COMMIT_MESSAGE}" || echo "لا تغييرات للـcommit"
    !git push origin {BRANCH}
    print("تم الرفع (أو لا توجد تغييرات)")

## ملاحظات

- إذا انقطعت الجلسة: أعد تشغيل الخلايا من **2** ثم **4** (resume تلقائي).
- لزيادة التدريب لاحقاً: ارفع `SCN_EPOCHS` فقط وأعد خلية التدريب.
- لا تشارك الدفتر وفيه التوكن ظاهر. ألغِ التوكن من GitHub بعد الانتهاء إن لزم.